# Unified Crop and Disease Corpus + ShareGPT Dataset Builder




## 1. Install dependencies

In [1]:
!pip install -q kaggle kagglehub "huggingface_hub[cli]>=0.24" datasets pillow tqdm requests pandas

## 2. Credentials setup


In [7]:
import os
import json
from google.colab import userdata
from huggingface_hub import login

# 1. API
kaggle_json_str = userdata.get('KAGGLE_API')
kaggle_data = json.loads(kaggle_json_str)

# 2. Save
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_data, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)


os.environ['KAGGLE_USERNAME'] = kaggle_data['username']
os.environ['KAGGLE_KEY'] = kaggle_data['key']


login()

## 3. Download source datasets

### 3.1 Kaggle dataset

In [5]:
import kagglehub

# kaggle download
kaggle_path = kagglehub.dataset_download(
    "snikhilrao/crop-disease-detection-dataset"
)
print(kaggle_path)

Using Colab cache for faster access to the 'crop-disease-detection-dataset' dataset.
/kaggle/input/crop-disease-detection-dataset


### 3.2 Mendeley dataset
Mendeley has no official SDK, so this hits their public files API. If the
endpoint changes, download the zip manually from the dataset page and
upload it, then unzip into `/content/mendeley_extracted`.

In [8]:
# mendeley skipped
EXTRACT_DIR = "/content/mendeley_extracted"
import os
os.makedirs(EXTRACT_DIR, exist_ok=True)  # stays empty

### 3.3 GitHub dataset (PlantDoc)

In [12]:
# clone plantdoc repo
!git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git /content/plantdoc

Cloning into '/content/plantdoc'...
remote: Enumerating objects: 2628, done.
remote: Counting objects: 100% (2628/2628), done.
remote: Compressing objects: 100% (2627/2627), done.
remote: Total 2628 (delta 1), reused 2616 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (2628/2628), 932.91 MiB | 30.10 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Updating files: 100% (2581/2581), done.


### 3.4 Hugging Face dataset

In [22]:
from google.colab import userdata
from huggingface_hub import login

# read token
token = userdata.get('HF_TOKEN')
login(token=token, add_to_git_credential=True)

print("Logged in successfully")

Logged in successfully


In [29]:
!pip uninstall -y hf_xet -q

In [1]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from google.colab import userdata
from huggingface_hub import snapshot_download

token = userdata.get('HF_TOKEN')

hf_path = snapshot_download(
    repo_id="Saon110/bd-crop-vegetable-plant-disease-dataset",
    repo_type="dataset",
    local_dir="/content/hf_bd_dataset",
    token=token,
)
print(hf_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

/content/hf_bd_dataset


## 4. Crop + disease keyword engine



In [2]:
CROPS = [
    "bell pepper", "chilli", "chili", "cauliflower", "cabbage", "cucumber",
    "eggplant", "brinjal", "groundnut", "pomegranate", "watermelon",
    "sugarcane", "strawberry", "blueberry", "raspberry", "soybean",
    "cassava", "coconut", "papaya", "mustard", "guava", "banana", "cherry",
    "grape", "peach", "apple", "maize", "corn", "wheat", "onion", "garlic",
    "ginger", "tomato", "potato", "pepper", "cotton", "squash", "orange",
    "lemon", "mango", "melon", "beans", "bean", "peas", "okra", "rice",
    "jute", "tea",
]
CROPS.sort(key=len, reverse=True)

# crop alias merge
CROP_ALIAS = {
    "chili": "chilli", "corn": "maize", "bean": "beans",
    "bell pepper": "pepper", "brinjal": "eggplant",
}

DISEASE_CANON = {
    "yellow leaf curl virus": "yellow leaf curl virus",
    "two spotted spider mite": "spider mites",
    "northern leaf blight": "northern leaf blight",
    "septoria leaf spot": "septoria leaf spot",
    "cedar apple rust": "cedar apple rust",
    "gray leaf spot": "gray leaf spot",
    "bacterial spot": "bacterial spot",
    "citrus greening": "citrus greening",
    "haunglongbing": "citrus greening",
    "powdery mildew": "powdery mildew",
    "downy mildew": "downy mildew",
    "mosaic virus": "mosaic virus",
    "common rust": "common rust",
    "early blight": "early blight",
    "late blight": "late blight",
    "spider mites": "spider mites",
    "black spot": "black spot",
    "leaf spot": "leaf spot",
    "leaf mold": "leaf mold",
    "target spot": "target spot",
    "black rot": "black rot",
    "leaf curl": "leaf curl",
    "anthracnose": "anthracnose",
    "healthy": "healthy",
    "mildew": "powdery mildew",
    "canker": "canker",
    "mosaic": "mosaic virus",
    "blight": "blight",
    "curl": "leaf curl",
    "scab": "scab",
    "rust": "rust",
    "wilt": "wilt",
    "rot": "rot",
}
DISEASE_KEYS = sorted(DISEASE_CANON.keys(), key=len, reverse=True)


def clean_text(text):
    # normalize separators
    t = text.lower()
    for ch in ("_", "-", "___", "__"):
        t = t.replace(ch, " ")
    return " ".join(t.split())


def extract_crop(text):
    # match crop keyword
    t = clean_text(text)
    for kw in CROPS:
        if kw in t:
            return CROP_ALIAS.get(kw, kw)
    return None


def extract_disease(text):
    # match disease keyword
    t = clean_text(text)
    for kw in DISEASE_KEYS:
        if kw in t:
            return DISEASE_CANON[kw]
    return None


def parse_label(text):
    # combine crop and disease
    crop = extract_crop(text)
    disease = extract_disease(text)
    if crop is None:
        return None
    if disease is None:
        disease = "unspecified"
    return crop, disease

## 5. Discover existing metadata (jsonl / csv / json)


In [3]:
import glob
import pandas as pd

FILE_COLS = ["file_name", "filename", "image", "image_path", "img_path", "path"]
LABEL_COLS = ["label", "class", "disease", "category", "class_name", "text", "caption"]


def find_metadata_files(root):
    # locate metadata files
    hits = []
    for ext in ("*.jsonl", "*.json", "*.csv"):
        hits += glob.glob(os.path.join(root, "**", ext), recursive=True)
    return hits


def load_table(path):
    # load one metadata file
    try:
        if path.endswith(".jsonl"):
            return pd.read_json(path, lines=True)
        if path.endswith(".json"):
            return pd.read_json(path)
        return pd.read_csv(path)
    except Exception:
        return None


def build_metadata_index(root):
    # build filename to label map
    index = {}
    for mpath in find_metadata_files(root):
        df = load_table(mpath)
        if df is None or df.empty:
            continue
        file_col = next((c for c in FILE_COLS if c in df.columns), None)
        label_col = next((c for c in LABEL_COLS if c in df.columns), None)
        if file_col is None or label_col is None:
            continue
        for _, row in df.iterrows():
            fname = os.path.basename(str(row[file_col]))
            index[fname] = str(row[label_col])
    return index

## 6. Walk sources, label every image



In [9]:
import uuid
from PIL import Image
from tqdm import tqdm

SOURCE_ROOTS = [kaggle_path, "/content/plantdoc", hf_path]
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

STAGE_DIR = "/content/staged"
os.makedirs(STAGE_DIR, exist_ok=True)

registry = []  # crop, disease, staged path, source


def process_root(root):
    # index then walk root
    meta_index = build_metadata_index(root)
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if not fn.lower().endswith(IMG_EXT):
                continue
            src_path = os.path.join(dirpath, fn)

            label_text = meta_index.get(fn)
            source_kind = "metadata"
            if label_text is None:
                label_text = dirpath + " " + fn
                source_kind = "foldername"

            parsed = parse_label(label_text)
            if parsed is None:
                continue
            crop, disease = parsed

            try:
                img = Image.open(src_path).convert("RGB")
            except Exception:
                continue

            out_name = f"{uuid.uuid4().hex}.jpg"
            out_path = os.path.join(STAGE_DIR, out_name)
            img.save(out_path, "JPEG", quality=90)

            registry.append({
                "staged_path": out_path,
                "crop": crop,
                "disease": disease,
                "label_source": source_kind,
                "origin": root,
            })


for root in tqdm(SOURCE_ROOTS, desc="sources"):
    process_root(root)

print("Total labeled images:", len(registry))
crops_seen = sorted({r["crop"] for r in registry})
print("Crops found:", crops_seen)

sources: 100%|██████████| 3/3 [10:43<00:00, 214.42s/it]

Total labeled images: 69696
Crops found: ['apple', 'beans', 'blueberry', 'cherry', 'cucumber', 'ginger', 'grape', 'maize', 'peach', 'pepper', 'potato', 'raspberry', 'soybean', 'squash', 'strawberry', 'tomato', 'wheat']


## 7. Balance classes, cap folder size, split



In [10]:
import random
from collections import defaultdict

random.seed(42)

MAX_PER_CLASS = 9000
MIN_PER_CLASS = 300
TARGET_TRAIN = 50000
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15

by_crop = defaultdict(list)
for r in registry:
    by_crop[r["crop"]].append(r)

# drop rare crops
kept = {c: v for c, v in by_crop.items() if len(v) >= MIN_PER_CLASS}
print("Kept crops:", len(kept), "of", len(by_crop))

capped_counts = {c: min(len(v), MAX_PER_CLASS) for c, v in kept.items()}
base_quota = min(capped_counts.values())
num_classes = len(kept)

ideal_train_each = -(-TARGET_TRAIN // num_classes)  # ceil div
per_class_train = min(base_quota, ideal_train_each, MAX_PER_CLASS)
per_class_val = max(1, int(per_class_train * VAL_FRACTION))
per_class_test = max(1, int(per_class_train * TEST_FRACTION))
per_class_total = min(per_class_train + per_class_val + per_class_test, MAX_PER_CLASS)

print("Per class -> train:", per_class_train, "val:", per_class_val,
      "test:", per_class_test, "total:", per_class_total)


def stratified_sample(entries, k):
    # round robin by disease
    buckets = defaultdict(list)
    for e in entries:
        buckets[e["disease"]].append(e)
    for b in buckets.values():
        random.shuffle(b)
    bucket_list = [b for b in buckets.values() if b]
    selected = []
    i = 0
    while len(selected) < k and bucket_list:
        b = bucket_list[i % len(bucket_list)]
        if b:
            selected.append(b.pop())
        if not b:
            bucket_list.remove(b) if b in bucket_list else None
        i += 1
        bucket_list = [b for b in bucket_list if b]
        if not bucket_list:
            break
    return selected[:k]

Kept crops: 9 of 17
Per class -> train: 4443 val: 666 test: 666 total: 5775


In [11]:
import shutil

FINAL_DIR = "/content/final_dataset"
for split in ("train", "val", "test"):
    os.makedirs(os.path.join(FINAL_DIR, split), exist_ok=True)

final_registry = []  # relative path, split, crop, disease

for crop, entries in kept.items():
    picked = stratified_sample(entries, per_class_total)
    random.shuffle(picked)

    train_e = picked[:per_class_train]
    val_e = picked[per_class_train:per_class_train + per_class_val]
    test_e = picked[per_class_train + per_class_val:]

    for split, split_entries in (("train", train_e), ("val", val_e), ("test", test_e)):
        out_dir = os.path.join(FINAL_DIR, split, crop)
        os.makedirs(out_dir, exist_ok=True)
        for e in split_entries:
            fname = os.path.basename(e["staged_path"])
            dst = os.path.join(out_dir, fname)
            shutil.copy2(e["staged_path"], dst)
            rel_path = os.path.join(split, crop, fname)
            final_registry.append({
                "image": rel_path,
                "split": split,
                "crop": crop,
                "disease": e["disease"],
                "label_source": e["label_source"],
            })

print("Final images:", len(final_registry))

Final images: 47489


In [12]:
# verify final counts
for split in ("train", "val", "test"):
    split_dir = os.path.join(FINAL_DIR, split)
    total = 0
    for cls in sorted(os.listdir(split_dir)):
        n = len(os.listdir(os.path.join(split_dir, cls)))
        total += n
        assert n <= 9000, f"{cls} exceeds cap"
    print(split, "total:", total)

train total: 39987
val total: 4172
test total: 3330


## 8. Save the structured label registry



In [13]:
import json

registry_path = os.path.join(FINAL_DIR, "labels_registry.jsonl")
with open(registry_path, "w") as f:
    for row in final_registry:
        f.write(json.dumps(row) + "\n")

print("Wrote", registry_path)

Wrote /content/final_dataset/labels_registry.jsonl


## 9. Build the ShareGPT-format dataset



In [14]:
def make_answer(crop, disease):
    # compose gpt answer
    crop_disp = crop.capitalize()
    if disease == "healthy":
        return f"This is a {crop_disp} leaf and it looks healthy, with no visible signs of disease."
    if disease == "unspecified":
        return f"This is a {crop_disp} leaf. It shows signs of stress, but the exact disease is unclear from the source label."
    return f"This is a {crop_disp} leaf affected by {disease}."


HUMAN_PROMPT = "<image>\nWhat crop is shown in this image, and does it have any disease? If so, name the disease."

sharegpt_rows = []
for i, row in enumerate(final_registry):
    sharegpt_rows.append({
        "id": f"{i:07d}",
        "image": row["image"],
        "conversations": [
            {"from": "human", "value": HUMAN_PROMPT},
            {"from": "gpt", "value": make_answer(row["crop"], row["disease"])},
        ],
    })

sharegpt_path = os.path.join(FINAL_DIR, "sharegpt.jsonl")
with open(sharegpt_path, "w") as f:
    for row in sharegpt_rows:
        f.write(json.dumps(row) + "\n")

print("Wrote", sharegpt_path, "-", len(sharegpt_rows), "conversations")
print(json.dumps(sharegpt_rows[0], indent=2))

Wrote /content/final_dataset/sharegpt.jsonl - 47489 conversations
{
  "id": "0000000",
  "image": "train/grape/03ce634c366c4b06b5a29dfecbafc2a4.jpg",
  "conversations": [
    {
      "from": "human",
      "value": "<image>\nWhat crop is shown in this image, and does it have any disease? If so, name the disease."
    },
    {
      "from": "gpt",
      "value": "This is a Grape leaf and it looks healthy, with no visible signs of disease."
    }
  ]
}


## 10. Load as an image dataset (sanity check)

In [15]:
from datasets import load_dataset

# load as imagefolder
ds = load_dataset(
    "imagefolder",
    data_dir=FINAL_DIR,
)
print(ds)

Resolving data files:   0%|          | 0/39987 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/4172 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/3330 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 39987
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 4172
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 3330
    })
})


## 11. Push everything to the Hub



In [16]:


# target repo id
REPO_ID = "w4ashabii/nepali_crop_data"



RepoUrl('https://huggingface.co/datasets/w4ashabii/nepali_crop_data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='w4ashabii/nepali_crop_data')

In [18]:
import os, zipfile

ZIP_DIR = "/content/upload_zips"
os.makedirs(ZIP_DIR, exist_ok=True)

MAX_ZIP_BYTES = 2 * 1024**3  # 2GB per zip

# collect every file to ship
all_files = []
for dirpath, _, filenames in os.walk(FINAL_DIR):
    for fn in filenames:
        full_path = os.path.join(dirpath, fn)
        rel_path = os.path.relpath(full_path, FINAL_DIR)
        all_files.append((full_path, rel_path))

print("Total files to zip:", len(all_files))

# pack files into batches
batches = []
current_batch = []
current_size = 0
for full_path, rel_path in all_files:
    size = os.path.getsize(full_path)
    if current_size + size > MAX_ZIP_BYTES and current_batch:
        batches.append(current_batch)
        current_batch = []
        current_size = 0
    current_batch.append((full_path, rel_path))
    current_size += size
if current_batch:
    batches.append(current_batch)

print("Number of zip batches:", len(batches))

# write each zip
zip_paths = []
for i, batch in enumerate(batches):
    zip_path = os.path.join(ZIP_DIR, f"dataset_part_{i:03d}.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for full_path, rel_path in batch:
            zf.write(full_path, arcname=rel_path)
    zip_paths.append(zip_path)
    print(zip_path, "-", len(batch), "files -",
          round(os.path.getsize(zip_path) / 1024**2, 1), "MB")

Total files to zip: 47491
Number of zip batches: 1
/content/upload_zips/dataset_part_000.zip - 47491 files - 1452.7 MB


In [19]:
from huggingface_hub import upload_file
import time

REPO_ID = "w4ashabii/nepali_crop_data"

for zp in zip_paths:
    name = os.path.basename(zp)
    for attempt in range(3):
        try:
            upload_file(
                path_or_fileobj=zp,
                path_in_repo=f"zips/{name}",
                repo_id=REPO_ID,
                repo_type="dataset",
                token=token,
            )
            print("Uploaded", name)
            break
        except Exception as e:
            print("Retrying", name, "after error:", e)
            time.sleep(5)
    else:
        print("FAILED to upload", name)

dataset_part_000.zip:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Uploaded dataset_part_000.zip
